Imports and paths

In [10]:
import pandas as pd
import numpy as np

TRAP_XLSX = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXR(from_Joe)/TRAP-PARDS_cxr.xlsx"
V1V2_XLSX = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
V3_XLSX   = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"
ANALYSIS_MASTER_CSV = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXRs_Raw/analysis_master.csv"

OUT_V1V2 = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V1V2.csv"
OUT_V3   = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V3.csv"

Helper functions

In [11]:
def to_str(s: pd.Series) -> pd.Series:
    """Convert to pandas string dtype (keeps NA)."""
    return s.astype("string")


def mrn_9(s: pd.Series) -> pd.Series:
    """
    Convert MRN to 9-digit zero-padded string.
    Handles int/float/string mixed input (Excel often causes floats like 12345.0).
    """
    s = s.astype("string")

    # Strip trailing .0 from excel float formatting, if present
    s = s.str.replace(r"\.0$", "", regex=True)

    # Keep digits only (in case of accidental formatting)
    s = s.str.replace(r"\D", "", regex=True)

    # Empty -> NA
    s = s.replace("", pd.NA)

    # Zero-pad to 9
    s = s.str.zfill(9)

    return s


def find_sheet_with_columns(xlsx_path: str, required_cols: set[str]) -> str:
    """
    Find the first sheet in an Excel file that contains all required columns.
    Useful because the 'Chest X-Ray Or...cession Number' sheet name is truncated.
    """
    xls = pd.ExcelFile(xlsx_path)
    for sh in xls.sheet_names:
        try:
            tmp = pd.read_excel(xlsx_path, sheet_name=sh, nrows=5)
        except Exception:
            continue
        cols = set(map(str, tmp.columns))
        if required_cols.issubset(cols):
            return sh
    raise RuntimeError(
        f"Could not find a sheet containing columns {sorted(required_cols)}.\n"
        f"Available sheets: {xls.sheet_names}"
    )

Load shared lookup tables once

In [12]:
# 1) TRAP PatientInfo (MRN <-> PatientID)
pi = pd.read_excel(TRAP_XLSX, sheet_name="PatientInfo")
required_pi = {"MRN", "PatientID"}
if not required_pi.issubset(set(pi.columns)):
    raise ValueError(f"TRAP PatientInfo missing {required_pi}. Columns: {pi.columns.tolist()}")

mrn_pid = (
    pi[["MRN", "PatientID"]]
    .assign(
        MRN=lambda d: mrn_9(d["MRN"]),
        PatientID=lambda d: to_str(d["PatientID"]),
    )
    .dropna(subset=["MRN", "PatientID"])
    .drop_duplicates()
    .reset_index(drop=True)
)

# 2) TRAP Chest accession sheet (PatientID <-> AccessionNumber)
acc_sheet = find_sheet_with_columns(TRAP_XLSX, {"PatientID", "AccessionNumber"})
trap_acc = pd.read_excel(TRAP_XLSX, sheet_name=acc_sheet)

pid_acc = (
    trap_acc[["PatientID", "AccessionNumber"]]
    .assign(
        PatientID=lambda d: to_str(d["PatientID"]),
        AccessionNumber=lambda d: to_str(d["AccessionNumber"]),  # ACC == AccessionNumber
    )
    .dropna(subset=["PatientID", "AccessionNumber"])
    .drop_duplicates()
    .reset_index(drop=True)
)

# 3) analysis_master ACC list (ACC == AccessionNumber)
am = pd.read_csv(ANALYSIS_MASTER_CSV)
if "ACC" not in am.columns:
    raise ValueError(f"analysis_master.csv missing ACC. Columns: {am.columns.tolist()}")

am_acc = (
    am[["ACC"]]
    .drop_duplicates()
    .assign(ACC=lambda d: to_str(d["ACC"]))
    .dropna(subset=["ACC"])
    .reset_index(drop=True)
)

print("Shared lookup tables loaded:")
print("  TRAP MRN<->PatientID rows:", len(mrn_pid), "| unique MRN:", mrn_pid["MRN"].nunique(), "| unique PatientID:", mrn_pid["PatientID"].nunique())
print("  TRAP PatientID<->AccessionNumber rows:", len(pid_acc), "| unique AccessionNumber:", pid_acc["AccessionNumber"].nunique(), "| sheet:", acc_sheet)
print("  analysis_master unique ACC:", len(am_acc))

Shared lookup tables loaded:
  TRAP MRN<->PatientID rows: 899 | unique MRN: 899 | unique PatientID: 899
  TRAP PatientID<->AccessionNumber rows: 10645 | unique AccessionNumber: 10645 | sheet: Chest X-Ray Or...cession Number
  analysis_master unique ACC: 2970


Reusable crosswalk builder

In [13]:
def build_crosswalk(vent_xlsx_path, vent_sheet_name, cohort_label, out_path):
    # -------------------------
    # Load vent MRNs
    # -------------------------
    vent = pd.read_excel(vent_xlsx_path, sheet_name=vent_sheet_name)
    if "MRN" not in vent.columns:
        raise ValueError(f"{cohort_label}: {vent_sheet_name} missing MRN. Columns: {vent.columns.tolist()}")

    vent_mrn = (
        vent[["MRN"]]
        .drop_duplicates()
        .assign(MRN=lambda d: mrn_9(d["MRN"]))
        .dropna(subset=["MRN"])
        .reset_index(drop=True)
    )

    # -------------------------
    # Base: all MRNs from this vent cohort
    # -------------------------
    cw = vent_mrn.copy()
    cw["cohort"] = cohort_label
    cw["vent_source_file"] = vent_xlsx_path
    cw["vent_source_sheet"] = vent_sheet_name
    cw["in_vent_cohort"] = True

    # -------------------------
    # MRN -> PatientID
    # -------------------------
    cw = cw.merge(mrn_pid, on="MRN", how="left")
    cw["in_trap_patientinfo"] = cw["PatientID"].notna()

    # -------------------------
    # PatientID -> AccessionNumber
    # -------------------------
    cw = cw.merge(pid_acc, on="PatientID", how="left")
    cw["in_trap_chest_sheet"] = cw["AccessionNumber"].notna()

    # -------------------------
    # AccessionNumber -> analysis_master ACC
    # -------------------------
    cw = cw.merge(
        am_acc.rename(columns={"ACC": "AccessionNumber"}).assign(in_analysis_master=True),
        on="AccessionNumber",
        how="left",
    )
    cw["in_analysis_master"] = cw["in_analysis_master"].fillna(False)

    # -------------------------
    # Diagnostics
    # -------------------------
    mrn_pid_counts = (
        mrn_pid.groupby("MRN")["PatientID"]
        .nunique()
        .reset_index(name="n_patientid_for_mrn")
    )

    pid_acc_counts = (
        pid_acc.groupby("PatientID")["AccessionNumber"]
        .nunique()
        .reset_index(name="n_accessions_for_patientid")
    )

    cw = cw.merge(mrn_pid_counts, on="MRN", how="left")
    cw = cw.merge(pid_acc_counts, on="PatientID", how="left")

    # -------------------------
    # Notes
    # -------------------------
    def make_notes(row):
        parts = []

        parts.append(f"MRN is present in {cohort_label} vent cohort ({vent_sheet_name}).")

        if row["in_trap_patientinfo"]:
            parts.append("PatientID found via TRAP-PARDS_cxr.xlsx / PatientInfo (MRN→PatientID).")
        else:
            parts.append("PatientID NOT found in TRAP-PARDS_cxr.xlsx / PatientInfo for this MRN.")

        if row["in_trap_chest_sheet"]:
            parts.append("AccessionNumber found via TRAP-PARDS_cxr.xlsx / Chest X-Ray Accession sheet (PatientID→AccessionNumber).")
        else:
            if row["in_trap_patientinfo"]:
                parts.append("No AccessionNumber found in TRAP-PARDS_cxr.xlsx / Chest X-Ray Accession sheet for this PatientID.")
            else:
                parts.append("AccessionNumber unavailable because PatientID is missing.")

        if pd.notna(row.get("AccessionNumber", pd.NA)):
            if row["in_analysis_master"]:
                parts.append("This AccessionNumber is present as ACC in analysis_master.csv (CXR exists in analysis_master).")
            else:
                parts.append("This AccessionNumber NOT found in analysis_master.csv ACC (no matching record in analysis_master).")

        if pd.notna(row.get("n_patientid_for_mrn", pd.NA)) and row["n_patientid_for_mrn"] and row["n_patientid_for_mrn"] > 1:
            parts.append(f"Warning: MRN maps to {int(row['n_patientid_for_mrn'])} PatientIDs (check TRAP PatientInfo consistency).")

        if pd.notna(row.get("n_accessions_for_patientid", pd.NA)) and row["n_accessions_for_patientid"] and row["n_accessions_for_patientid"] > 1:
            parts.append(f"Info: PatientID has {int(row['n_accessions_for_patientid'])} accession numbers.")

        return " ".join(parts)

    cw["Notes"] = cw.apply(make_notes, axis=1)

    # Safety: keep MRN as 9-digit string
    cw["MRN"] = mrn_9(cw["MRN"])

    # Reorder columns
    col_order = [
        "MRN",
        "PatientID",
        "AccessionNumber",
        "cohort",
        "vent_source_file",
        "vent_source_sheet",
        "in_vent_cohort",
        "in_trap_patientinfo",
        "in_trap_chest_sheet",
        "in_analysis_master",
        "n_patientid_for_mrn",
        "n_accessions_for_patientid",
        "Notes",
    ]
    cw = cw[[c for c in col_order if c in cw.columns]]

    # Save
    cw.to_csv(out_path, index=False)

    # Summary
    print(f"\nSaved: {out_path}")
    print("Rows:", cw.shape[0], "Cols:", cw.shape[1])
    print("Summary:")
    print("  Cohort:", cohort_label)
    print("  Vent MRNs (unique):", cw["MRN"].nunique())
    print("  MRNs with PatientID:", cw.groupby("MRN")["PatientID"].apply(lambda x: x.notna().any()).sum())
    print("  MRNs with >=1 AccessionNumber:", cw.groupby("MRN")["AccessionNumber"].apply(lambda x: x.notna().any()).sum())
    print("  Rows where AccessionNumber is found in analysis_master:", int(cw["in_analysis_master"].sum()))
    print("  All MRNs length==9:", bool((cw["MRN"].str.len() == 9).all()))

    return cw

Build V1V2 crosswalk using Sheet5

In [14]:
cw_v1v2 = build_crosswalk(
    vent_xlsx_path=V1V2_XLSX,
    vent_sheet_name="Sheet5",
    cohort_label="V1V2",
    out_path=OUT_V1V2,
)

/tmp/ipykernel_3974316/1765118848.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cw["in_analysis_master"] = cw["in_analysis_master"].fillna(False)



Saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V1V2.csv
Rows: 3874 Cols: 13
Summary:
  Cohort: V1V2
  Vent MRNs (unique): 317
  MRNs with PatientID: 314
  MRNs with >=1 AccessionNumber: 292
  Rows where AccessionNumber is found in analysis_master: 1552
  All MRNs length==9: True


Build V3 crosswalk using Sheet15

In [15]:
cw_v3 = build_crosswalk(
    vent_xlsx_path=V3_XLSX,
    vent_sheet_name="Sheet15",
    cohort_label="V3",
    out_path=OUT_V3,
)


Saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V3.csv
Rows: 236 Cols: 13
Summary:
  Cohort: V3
  Vent MRNs (unique): 63
  MRNs with PatientID: 63
  MRNs with >=1 AccessionNumber: 9
  Rows where AccessionNumber is found in analysis_master: 34
  All MRNs length==9: True


/tmp/ipykernel_3974316/1765118848.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cw["in_analysis_master"] = cw["in_analysis_master"].fillna(False)


Optional quick comparison

In [16]:
print("\nQuick comparison:")
print("V1V2 unique MRNs:", cw_v1v2["MRN"].nunique())
print("V3   unique MRNs:", cw_v3["MRN"].nunique())

print("\nV1V2 rows with analysis_master match:", int(cw_v1v2["in_analysis_master"].sum()))
print("V3   rows with analysis_master match:", int(cw_v3["in_analysis_master"].sum()))


Quick comparison:
V1V2 unique MRNs: 317
V3   unique MRNs: 63

V1V2 rows with analysis_master match: 1552
V3   rows with analysis_master match: 34


### Vent window with best CXRs

Imports and paths

In [23]:
import pandas as pd
import numpy as np

# Vent files
V1V2_XLSX = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1V2_df.xlsx"
V3_XLSX   = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

# Crosswalk files you already created
CW_V1V2_CSV = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V1V2.csv"
CW_V3_CSV   = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/overall_crosswalk_mrn_patientid_acc_notes_V3.csv"

# CXR master
ANALYSIS_MASTER_CSV = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/2025_CXRs_Raw/analysis_master.csv"

# Outputs
OUT_V1V2 = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/vent_window_with_best_cxr_V1V2.csv"
OUT_V3   = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/vent_window_with_best_cxr_V3.csv"

Helper functions

In [20]:
def mrn_9(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\D", "", regex=True)
    s = s.replace("", pd.NA)
    s = s.str.zfill(9)
    return s


def to_str(s: pd.Series) -> pd.Series:
    return s.astype("string")


def parse_dicom_datetime(date_col: pd.Series, time_col: pd.Series) -> pd.Series:
    """
    Parse DICOM StudyDate + StudyTime into pandas Timestamp.
    Handles forms like 20250606.0 and 173556.0 / 175859.867
    """
    out = []
    for d, t in zip(date_col, time_col):
        if pd.isna(d) or pd.isna(t):
            out.append(pd.NaT)
            continue
        try:
            d_str = str(int(float(d))).zfill(8)

            t_float = float(t)
            hh = int(t_float // 10000)
            mm = int((t_float - hh * 10000) // 100)
            ss = t_float - hh * 10000 - mm * 100
            sec = int(ss)
            micro = int(round((ss - sec) * 1_000_000))

            ts = pd.Timestamp(
                year=int(d_str[:4]),
                month=int(d_str[4:6]),
                day=int(d_str[6:8]),
                hour=hh,
                minute=mm,
                second=sec,
                microsecond=micro,
            )
            out.append(ts)
        except Exception:
            out.append(pd.NaT)
    return pd.Series(out)


def load_analysis_master(path):
    am = pd.read_csv(path)

    # Standardize ACC
    am["ACC"] = to_str(am["ACC"])

    # Parse datetime
    if "StudyDate" in am.columns and "StudyTime" in am.columns:
        am["StudyDateTime"] = parse_dicom_datetime(am["StudyDate"], am["StudyTime"])
    else:
        am["StudyDateTime"] = pd.NaT

    return am

CXR selection helper

In [31]:
def select_best_cxr_for_vent(
    vent_df,
    crosswalk_df,
    analysis_master_df,
    vent_time_col="1st_Time_Start",
    max_abs_hours=24,
    frontal_only=False,
):
    """
    Returns one row per vent row, with the best matching CXR attached.
    """

    vent = vent_df.copy()
    cw = crosswalk_df.copy()
    am = analysis_master_df.copy()

    # Standardize MRN
    vent["MRN"] = mrn_9(vent["MRN"])
    cw["MRN"] = mrn_9(cw["MRN"])

    # Standardize vent time
    vent[vent_time_col] = pd.to_datetime(vent[vent_time_col], errors="coerce")

    # Standardize accession columns
    cw["AccessionNumber"] = to_str(cw["AccessionNumber"])
    am["ACC"] = to_str(am["ACC"])

    # Keep only rows marked present in analysis_master if available
    if "in_analysis_master" in cw.columns:
        cw = cw[cw["in_analysis_master"].astype(str).str.lower() == "true"].copy()

    # Merge crosswalk with analysis_master
    # IMPORTANT: because analysis_master also has an AccessionNumber column,
    # pandas will create AccessionNumber_x / AccessionNumber_y
    cxr = cw.merge(
        am,
        left_on="AccessionNumber",
        right_on="ACC",
        how="inner",
        suffixes=("_cw", "_am")
    )

    # Rename accession columns clearly
    if "AccessionNumber_cw" in cxr.columns:
        cxr = cxr.rename(columns={"AccessionNumber_cw": "AccessionNumber"})
    elif "AccessionNumber_x" in cxr.columns:
        cxr = cxr.rename(columns={"AccessionNumber_x": "AccessionNumber"})

    # Optional frontal filter
    if frontal_only and "ViewPosition" in cxr.columns:
        cxr = cxr[cxr["ViewPosition"].isin(["AP", "PA"])].copy()

    # Keep useful columns
    preferred_cxr_cols = [
        "MRN",
        "PatientID",
        "AccessionNumber",   # from crosswalk
        "ACC",               # from analysis_master
        "StudyDateTime",
        "dicom_path",
        "filename",
        "ViewPosition",
        "Modality",
        "StudyInstanceUID",
        "SeriesInstanceUID",
        "SOPInstanceUID",
        "Rows",
        "Columns",
        "QC_missing_spacing",
        "QC_missing_dims",
    ]
    keep_cxr_cols = [c for c in preferred_cxr_cols if c in cxr.columns]
    cxr = cxr[keep_cxr_cols].copy()

    # Add unique vent row id
    vent = vent.reset_index(drop=True).copy()
    vent["vent_row_id"] = np.arange(len(vent))

    # Merge all candidate CXRs onto each vent row by MRN
    merged = vent.merge(cxr, on="MRN", how="left")

    # Compute delta time
    merged["delta_hours"] = (
        (pd.to_datetime(merged["StudyDateTime"], errors="coerce") - merged[vent_time_col])
        .dt.total_seconds() / 3600.0
    )
    merged["abs_delta_hours"] = merged["delta_hours"].abs()

    # Sort best candidate first
    merged = merged.sort_values(
        by=["vent_row_id", "abs_delta_hours", "StudyDateTime"],
        ascending=[True, True, True],
        na_position="last"
    )

    # Keep best CXR per vent row
    best = merged.groupby("vent_row_id", as_index=False).first()

    # Flags
    best["has_cxr_match"] = best["AccessionNumber"].notna()
    best[f"within_{int(max_abs_hours)}h"] = best["abs_delta_hours"] <= max_abs_hours
    best[f"within_{int(max_abs_hours)}h"] = best[f"within_{int(max_abs_hours)}h"].fillna(False)

    # Match note
    def make_match_note(row):
        if not row["has_cxr_match"]:
            return "No CXR accession found in analysis_master for this vent sample."
        if pd.isna(row["delta_hours"]):
            return "CXR accession found but StudyDateTime is missing or unparsable."
        if row[f"within_{int(max_abs_hours)}h"]:
            return f"Best CXR found; |delta_hours| <= {max_abs_hours}."
        return f"Best CXR found, but outside {max_abs_hours}h."

    best["cxr_match_note"] = best.apply(make_match_note, axis=1)

    # Put original vent columns first
    vent_cols = list(vent_df.columns)
    extra_cols = [c for c in best.columns if c not in vent_cols and c != "vent_row_id"]
    best = best[vent_cols + extra_cols]

    return best

Load the datasets

In [32]:
# Load vent cohorts
vent_v1v2 = pd.read_excel(V1V2_XLSX, sheet_name="Sheet5")
vent_v3   = pd.read_excel(V3_XLSX, sheet_name="Sheet15")

# Load crosswalks
cw_v1v2 = pd.read_csv(CW_V1V2_CSV, dtype=str)
cw_v3   = pd.read_csv(CW_V3_CSV, dtype=str)

# Load analysis_master
am = load_analysis_master(ANALYSIS_MASTER_CSV)

print("Loaded:")
print("  vent_v1v2:", vent_v1v2.shape)
print("  vent_v3  :", vent_v3.shape)
print("  cw_v1v2  :", cw_v1v2.shape)
print("  cw_v3    :", cw_v3.shape)
print("  am       :", am.shape)

Loaded:
  vent_v1v2: (432, 1840)
  vent_v3  : (65, 1844)
  cw_v1v2  : (3874, 13)
  cw_v3    : (236, 13)
  am       : (3124, 100)


Build vent_window_with_best_cxr_V1V2.csv

In [33]:
matched_v1v2 = select_best_cxr_for_vent(
    vent_df=vent_v1v2,
    crosswalk_df=cw_v1v2,
    analysis_master_df=am,
    vent_time_col="1st_Time_Start",
    max_abs_hours=24,
    frontal_only=False,
)

matched_v1v2.to_csv(OUT_V1V2, index=False)

print("Saved:", OUT_V1V2)
print("Total vent rows:", len(matched_v1v2))
print("Rows with CXR match:", int(matched_v1v2["has_cxr_match"].sum()))
print("Rows within 24h:", int(matched_v1v2["within_24h"].sum()))
print("Match rate:", float(matched_v1v2["has_cxr_match"].mean()))
print("Within-24h rate:", float(matched_v1v2["within_24h"].mean()))

Saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/vent_window_with_best_cxr_V1V2.csv
Total vent rows: 432
Rows with CXR match: 399
Rows within 24h: 369
Match rate: 0.9236111111111112
Within-24h rate: 0.8541666666666666


Build vent_window_with_best_cxr_V3.csv

In [34]:
matched_v3 = select_best_cxr_for_vent(
    vent_df=vent_v3,
    crosswalk_df=cw_v3,
    analysis_master_df=am,
    vent_time_col="1st_Time_Start",
    max_abs_hours=24,
    frontal_only=False,
)

matched_v3.to_csv(OUT_V3, index=False)

print("Saved:", OUT_V3)
print("Total vent rows:", len(matched_v3))
print("Rows with CXR match:", int(matched_v3["has_cxr_match"].sum()))
print("Rows within 24h:", int(matched_v3["within_24h"].sum()))
print("Match rate:", float(matched_v3["has_cxr_match"].mean()))
print("Within-24h rate:", float(matched_v3["within_24h"].mean()))

Saved: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/vent_window_with_best_cxr_V3.csv
Total vent rows: 65
Rows with CXR match: 7
Rows within 24h: 0
Match rate: 0.1076923076923077
Within-24h rate: 0.0


/tmp/ipykernel_3974316/2594054643.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  best = merged.groupby("vent_row_id", as_index=False).first()


Quick summaries you should check

In [35]:
def summarize_matched(df, cohort_name):
    print(f"\n=== {cohort_name} ===")
    print("Rows:", len(df))
    print("Unique MRN:", df["MRN"].astype(str).nunique())

    print("Rows with any CXR match:", int(df["has_cxr_match"].sum()))
    print("Rows within 24h:", int(df["within_24h"].sum()))

    if "delta_hours" in df.columns:
        tmp = df.loc[df["has_cxr_match"] & df["delta_hours"].notna(), "delta_hours"]
        if len(tmp) > 0:
            print("Delta hours summary:")
            print(tmp.describe())


summarize_matched(matched_v1v2, "V1V2")
summarize_matched(matched_v3, "V3")


=== V1V2 ===
Rows: 432
Unique MRN: 317
Rows with any CXR match: 399
Rows within 24h: 369
Delta hours summary:
count      399.000000
mean       -57.090572
std        696.326128
min     -10777.013056
25%         -0.979861
50%          0.297222
75%          0.912222
max        143.528611
Name: delta_hours, dtype: float64

=== V3 ===
Rows: 65
Unique MRN: 63
Rows with any CXR match: 7
Rows within 24h: 0
Delta hours summary:
count       7.000000
mean    -6167.767897
std      1162.801857
min     -7249.981389
25%     -7215.646806
50%     -6380.881389
75%     -5311.178056
max     -4489.862778
Name: delta_hours, dtype: float64
